# 07 — Baseline Model: Logistic Regression

**Owner:** Member 2 — Narendra Iyer 
**Project:** AeroDelay AI — Airline Delay Risk Prediction 
**Course:** AAI-540, Group 3

## Purpose
Train a Logistic Regression baseline using a SageMaker Training Job.  
This establishes the performance floor that our XGBoost model must beat.

## Prerequisites
- Notebooks `01` through `05` have been run successfully by Member 1 (Aleena).
- Processed parquet files exist in S3 under `airline-delay/training/`, `airline-delay/validation/`, `airline-delay/test/`.
- `%store` variables `s3_aerodelay` is available from notebook 01.

## 1. Setup

In [1]:
import boto3, os, json, time
import pandas as pd
import awswrangler as wr
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess   = Session()
bucket = sess.default_bucket()
role   = get_execution_role()
region = sess.boto_region_name
s3     = boto3.client("s3")
sm     = boto3.client("sagemaker")

print(f"Bucket : {bucket}")
print(f"Region : {region}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Bucket : sagemaker-us-east-1-151132426745
Region : us-east-1


In [2]:
# Restore shared variable from notebook 01
%store -r s3_aerodelay
print(f"s3_aerodelay: {s3_aerodelay}")

s3_aerodelay: s3://sagemaker-us-east-1-151132426745/airline-delay


## 2. Verify Training Data in S3

Confirm Member 1's preprocessing outputs are available before launching any training jobs.

In [3]:
for split in ["training", "validation", "test", "production_simulation"]:
    prefix = f"airline-delay/{split}/data.parquet"
    try:
        resp    = s3.head_object(Bucket=bucket, Key=prefix)
        size_mb = resp["ContentLength"] / 1024 / 1024
        print(f"  {split}: {size_mb:.2f} MB")
    except Exception as e:
        print(f"  {split}: NOT FOUND — {e}")

  training: 8.61 MB
  validation: 2.24 MB
  test: 2.18 MB
  production_simulation: 8.62 MB


In [4]:
# Quick peek at training data shape and class balance
train_df = wr.s3.read_parquet(f"{s3_aerodelay}/training/data.parquet")
print(f"Train shape : {train_df.shape}")
print(f"Delay rate  : {train_df['arrdel15'].mean():.1%}")
print(f"Features    : {[c for c in train_df.columns if c != 'arrdel15']}")
train_df.head(3)

2026-06-13 17:13:56,544	WARNING services.py:2137 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 408924160 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=0.74gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.


2026-06-13 17:13:57,730	INFO worker.py:2007 -- Started a local Ray instance.


/opt/conda/lib/python3.12/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Train shape : (650420, 20)
Delay rate  : 22.5%
Features    : ['year', 'month', 'dayofmonth', 'dayofweek', 'reporting_airline', 'origin', 'dest', 'originstate', 'deststate', 'crselapsedtime', 'distance', 'dep_hour', 'is_weekend', 'route', 'carrier_delay_rate', 'origin_delay_rate', 'dest_delay_rate', 'route_delay_rate', 'hour_delay_rate']


,year,month,dayofmonth,dayofweek,reporting_airline,origin,dest,originstate,deststate,crselapsedtime,distance,arrdel15,dep_hour,is_weekend,route,carrier_delay_rate,origin_delay_rate,dest_delay_rate,route_delay_rate,hour_delay_rate
0,2024,1,1,1,12,SPN,GUM,42,42,45.0,129.0,0,9,0,SPN_GUM,0.207039,0.026316,0.131579,0.026316,0.191854
1,2024,1,1,1,6,PIE,ATW,7,49,182.0,1173.0,0,16,0,PIE_ATW,0.207776,0.151899,0.286517,0.125000,0.267850
2,2024,1,1,1,6,GRR,SFB,20,7,164.0,1001.0,0,12,0,GRR_SFB,0.207776,0.283306,0.217822,0.200000,0.227549


## 3. Write the Baseline Training Script

SageMaker Training Jobs require a self-contained Python script.
We write it locally to `src/training/train_baseline.py` — the same
folder structure recommended in the project plan.

In [5]:
os.makedirs("src/training", exist_ok=True)

baseline_script = '''
# train_baseline.py — Logistic Regression baseline for AeroDelay AI
# Runs inside a SageMaker Training Job container.
import argparse, os, json
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

TARGET = "arrdel15"

def load_parquet(path):
    files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith(".parquet")]
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

def evaluate(model, X, y, split_name):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    metrics = {
        "split"     : split_name,
        "accuracy"  : round(float(accuracy_score(y, y_pred)), 4),
        "precision" : round(float(precision_score(y, y_pred, zero_division=0)), 4),
        "recall"    : round(float(recall_score(y, y_pred, zero_division=0)), 4),
        "f1"        : round(float(f1_score(y, y_pred, zero_division=0)), 4),
        "roc_auc"   : round(float(roc_auc_score(y, y_prob)), 4),
    }
    cm = confusion_matrix(y, y_pred).tolist()
    print(json.dumps({"metrics": metrics, "confusion_matrix": cm}))
    return metrics, cm

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--train",           default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--validation",      default=os.environ.get("SM_CHANNEL_VALIDATION"))
    parser.add_argument("--model-dir",       default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--output-data-dir", default=os.environ.get("SM_OUTPUT_DATA_DIR"))
    parser.add_argument("--max-iter",        type=int,   default=300)
    parser.add_argument("--C",               type=float, default=1.0)
    parser.add_argument("--solver",          default="lbfgs")
    args = parser.parse_args()

    print("Loading data...")
    train_df = load_parquet(args.train)
    val_df   = load_parquet(args.validation)

    feature_cols = [c for c in train_df.columns if c != TARGET]
    X_train = train_df[feature_cols].fillna(0)
    y_train = train_df[TARGET]
    X_val   = val_df[feature_cols].fillna(0)
    y_val   = val_df[TARGET]

    print(f"Train : {X_train.shape} | delay rate {y_train.mean():.3f}")
    print(f"Val   : {X_val.shape}   | delay rate {y_val.mean():.3f}")
    print(f"Features: {feature_cols}")

    # StandardScaler is important for Logistic Regression
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            max_iter=args.max_iter,
            C=args.C,
            solver=args.solver,
            class_weight="balanced",   # handles 4:1 class imbalance
            n_jobs=-1,
            random_state=42
        ))
    ])

    print("Training Logistic Regression baseline...")
    model.fit(X_train, y_train)

    train_metrics, train_cm = evaluate(model, X_train, y_train, "train")
    val_metrics,   val_cm   = evaluate(model, X_val,   y_val,   "validation")

    os.makedirs(args.output_data_dir, exist_ok=True)
    output = {
        "model"      : "LogisticRegression",
        "train"      : {"metrics": train_metrics, "confusion_matrix": train_cm},
        "validation" : {"metrics": val_metrics,   "confusion_matrix": val_cm},
        "features"   : feature_cols,
        "hyperparameters": {"max_iter": args.max_iter, "C": args.C, "solver": args.solver}
    }
    with open(os.path.join(args.output_data_dir, "baseline_metrics.json"), "w") as f:
        json.dump(output, f, indent=2)

    joblib.dump(model, os.path.join(args.model_dir, "model.joblib"))
    joblib.dump(feature_cols, os.path.join(args.model_dir, "feature_cols.joblib"))
    print("Model and feature list saved.")
'''

with open("src/training/train_baseline.py", "w") as f:
    f.write(baseline_script.strip())

print("Written: src/training/train_baseline.py")

Written: src/training/train_baseline.py


In [7]:
import sagemaker as sm_sdk
sess.sagemaker_config = sm_sdk.Session().sagemaker_config

## 4. Launch Baseline SageMaker Training Job

In [11]:
os.makedirs("src/training", exist_ok=True)

baseline_script = '''
import argparse, os, json
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

TARGET = "arrdel15"

def load_parquet(path):
    files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith(".parquet")]
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

def encode_strings(df):
    for col in df.select_dtypes(include=["object", "string"]).columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    return df

def evaluate(model, X, y, split_name):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    metrics = {
        "split"     : split_name,
        "accuracy"  : round(float(accuracy_score(y, y_pred)), 4),
        "precision" : round(float(precision_score(y, y_pred, zero_division=0)), 4),
        "recall"    : round(float(recall_score(y, y_pred, zero_division=0)), 4),
        "f1"        : round(float(f1_score(y, y_pred, zero_division=0)), 4),
        "roc_auc"   : round(float(roc_auc_score(y, y_prob)), 4),
    }
    cm = confusion_matrix(y, y_pred).tolist()
    print(json.dumps({"metrics": metrics, "confusion_matrix": cm}))
    return metrics, cm

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--train",           default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--validation",      default=os.environ.get("SM_CHANNEL_VALIDATION"))
    parser.add_argument("--model-dir",       default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--output-data-dir", default=os.environ.get("SM_OUTPUT_DATA_DIR"))
    parser.add_argument("--max-iter",        type=int,   default=300)
    parser.add_argument("--solver",          default="lbfgs")
    args = parser.parse_args()

    print("Loading data...")
    train_df = load_parquet(args.train)
    val_df   = load_parquet(args.validation)

    feature_cols = [c for c in train_df.columns if c != TARGET]
    X_train = encode_strings(train_df[feature_cols].fillna(0))
    y_train = train_df[TARGET]
    X_val   = encode_strings(val_df[feature_cols].fillna(0))
    y_val   = val_df[TARGET]

    print(f"Train : {X_train.shape} | delay rate {y_train.mean():.3f}")
    print(f"Val   : {X_val.shape}   | delay rate {y_val.mean():.3f}")

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            max_iter=args.max_iter,
            C=1.0,
            solver=args.solver,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        ))
    ])

    print("Training Logistic Regression baseline...")
    model.fit(X_train, y_train)

    train_metrics, train_cm = evaluate(model, X_train, y_train, "train")
    val_metrics,   val_cm   = evaluate(model, X_val,   y_val,   "validation")

    os.makedirs(args.output_data_dir, exist_ok=True)
    output = {
        "model"      : "LogisticRegression",
        "train"      : {"metrics": train_metrics, "confusion_matrix": train_cm},
        "validation" : {"metrics": val_metrics,   "confusion_matrix": val_cm},
        "features"   : feature_cols,
        "hyperparameters": {"max_iter": args.max_iter, "solver": args.solver}
    }
    with open(os.path.join(args.output_data_dir, "baseline_metrics.json"), "w") as f:
        json.dump(output, f, indent=2)

    joblib.dump(model, os.path.join(args.model_dir, "model.joblib"))
    joblib.dump(feature_cols, os.path.join(args.model_dir, "feature_cols.joblib"))
    print("Model and feature list saved.")
'''

with open("src/training/train_baseline.py", "w") as f:
    f.write(baseline_script.strip())

print("Written: src/training/train_baseline.py")

Written: src/training/train_baseline.py


In [12]:
import sagemaker
from sagemaker.sklearn.estimator import SKLearn

standard_sess = sagemaker.Session()
standard_role = sagemaker.get_execution_role()

baseline_estimator = SKLearn(
    entry_point       = "train_baseline.py",
    source_dir        = "src/training",
    framework_version = "1.2-1",
    instance_type     = "ml.m5.large",
    instance_count    = 1,
    role              = standard_role,
    sagemaker_session = standard_sess,
    base_job_name     = "aerodelay-baseline",
    hyperparameters   = {"max-iter": 300, "solver": "lbfgs"},
    output_path       = f"{s3_aerodelay}/model-artifacts/baseline/",
)

baseline_estimator.fit(
    inputs={
        "train"      : f"{s3_aerodelay}/training/",
        "validation" : f"{s3_aerodelay}/validation/",
    },
    wait=True,
    logs=True
)

print("\nBaseline training job complete.")
print("Job name  :", baseline_estimator.latest_training_job.name)
print("Model URI :", baseline_estimator.model_data)

INFO:sagemaker:Creating training-job with name: aerodelay-baseline-2026-06-13-17-38-24-100


2026-06-13 17:38:25 Starting - Starting the training job.

.

.


2026-06-13 17:38:41 Starting - Preparing the instances for training.

...

.

.

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-06-13 17:40:33,460 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-06-13 17:40:33,464 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-06-13 17:40:33,467 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-06-13 17:40:33,484 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-06-13 17:40:33,776 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-06-13 17:40:33,780 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-06-


2026-06-13 17:41:00 Training - Training image download completed. Training in progress.
2026-06-13 17:41:00 Uploading - Uploading generated training model
2026-06-13 17:41:00 Completed - Training job completed


Training seconds: 120
Billable seconds: 120

Baseline training job complete.
Job name  : aerodelay-baseline-2026-06-13-17-38-24-100
Model URI : s3://sagemaker-us-east-1-151132426745/airline-delay/model-artifacts/baseline/aerodelay-baseline-2026-06-13-17-38-24-100/output/model.tar.gz


## 5. Store Job Details for Downstream Notebooks

In [13]:
baseline_job_name  = baseline_estimator.latest_training_job.name
baseline_model_uri = baseline_estimator.model_data

%store baseline_job_name
%store baseline_model_uri

print(f"baseline_job_name  : {baseline_job_name}")
print(f"baseline_model_uri : {baseline_model_uri}")

Stored 'baseline_job_name' (str)
Stored 'baseline_model_uri' (str)
baseline_job_name  : aerodelay-baseline-2026-06-13-17-38-24-100
baseline_model_uri : s3://sagemaker-us-east-1-151132426745/airline-delay/model-artifacts/baseline/aerodelay-baseline-2026-06-13-17-38-24-100/output/model.tar.gz


## 6. Summary - Baseline Model Results

I established a Logistic Regression baseline in this notebook using a SageMaker Training Job running on an `ml.m5.large` instance. Before launching the training job, I verified that all four data splits created by Member 1 (Aleena) were available in S3 — training (8.61 MB), validation (2.24 MB), test (2.18 MB), and production simulation (8.62 MB).

**Training Data Overview:**
- Total training records: 650,420 flights
- Delay rate: 22.5% — confirming the 4:1 class imbalance identified during EDA
- Features: 19 engineered features covering temporal, geographic, carrier, and historical delay rate signals

**Model Design:**
The baseline uses a `StandardScaler + LogisticRegression` pipeline with `class_weight="balanced"` to compensate for the class imbalance. This is intentional — a model that ignores class weights would simply predict "on time" for most flights and achieve high accuracy without any real predictive value.

**Training Job:**
- Job name: `aerodelay-baseline-2026-06-13-17-38-24-100`
- Instance: `ml.m5.large`
- Billable time: 120 seconds
- Model artifact stored in S3 under `airline-delay/model-artifacts/baseline/`

The baseline job name and model URI are stored via `%store` so notebook 09 can load this model for head-to-head comparison against XGBoost. The purpose of this baseline is to set a performance floor — if XGBoost cannot beat these numbers on recall, F1, and ROC-AUC, it would not be worth the additional complexity and compute cost.